In [13]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [14]:
# ============================================================== 
# Parámetros generales
# ============================================================== 
CSV_GAME = "../../../data/inputs/csv/game.csv"
CSV_LINE = "../../../data/inputs/csv/line_score.csv"
CSV_OTHER = "../../../data/inputs/csv/other_stats.csv"
RANDOM_STATE = 42
EWMA_SPAN = 8
ELO_K = 20
INITIAL_ELO = 1500
DECAY_FACTOR = 0.98

In [15]:
# 1:Cargar datos
games = pd.read_csv(CSV_GAME)
line_score = pd.read_csv(CSV_LINE)
#ordena los partidos cronologicamente
games['game_date'] = pd.to_datetime(games['game_date'], errors='coerce')
games = games.sort_values('game_date').reset_index(drop=True)
games['game_id'] = games.index
#define la variable home_win
games['home_win'] = (games['pts_home'] > games['pts_away']).astype(int)

FileNotFoundError: [Errno 2] No such file or directory: '../../../data/inputs/csv/game.csv'

In [16]:
# 2:ELO. 
#sistema heredado de los modelos de ajedrez y que consiste en actualizar la "fuerza" de un
#equipo tras cada partidoen función de la prevision y el resultado
#Cuanto más improbable sea la victoria (según el rating previo), mayor la recompensa
teams = pd.unique(games[['team_id_home', 'team_id_away']].values.ravel())
elo = {t: INITIAL_ELO for t in teams}
elo_history = []
for idx, row in games.iterrows():
    h, a = row['team_id_home'], row['team_id_away']
    he, ae = elo[h], elo[a]
    elo_history.append((he, ae))
    exp_h = 1.0 / (1.0 + 10 ** ((ae - he) / 400.0))
    actual_h = 1.0 if row['pts_home'] > row['pts_away'] else 0.0
    elo[h] += ELO_K * (actual_h - exp_h)
    elo[a] += ELO_K * ((1 - actual_h) - (1 - exp_h))

games['elo_home'] = [e[0] for e in elo_history]
games['elo_away'] = [e[1] for e in elo_history]
games['elo_diff'] = games['elo_home'] - games['elo_away']

NameError: name 'games' is not defined

In [ ]:
#3:Procesa line_score
try:
    pts_cols_home = [c for c in line_score.columns if c.startswith('pts_qrt') and c.endswith('_home')]
    pts_cols_away = [c for c in line_score.columns if c.startswith('pts_qrt') and c.endswith('_away')]

    if pts_cols_home and pts_cols_away:
        line_score['home_total_pts'] = line_score[pts_cols_home].sum(axis=1)
        line_score['away_total_pts'] = line_score[pts_cols_away].sum(axis=1)
        #diff_pts_line: puntos totales locales-visitantes
        line_score['diff_pts_line'] = line_score['home_total_pts'] - line_score['away_total_pts']
    else:
        line_score['diff_pts_line'] = 0
    games = games.merge(line_score[['game_id', 'diff_pts_line']], on='game_id', how='left')
except Exception as e:
    print("⚠️ line_score no se usó correctamente:", e)
    games['diff_pts_line'] = 0


In [ ]:
#4:EWMA + STREAK
#mide la tendencia de un quipo en función de sus ultimos resultados
home_cols = ['game_id', 'game_date', 'team_id_home', 'pts_home', 'fgm_home', 'fga_home',
             'fg3m_home', 'fg3a_home', 'ftm_home', 'fta_home', 'reb_home', 'ast_home']
away_cols = ['game_id', 'game_date', 'team_id_away', 'pts_away', 'fgm_away', 'fga_away',
             'fg3m_away', 'fg3a_away', 'ftm_away', 'fta_away', 'reb_away', 'ast_away']

home_df = games[home_cols].copy().rename(columns={
    'team_id_home': 'team_id', 'pts_home': 'pts', 'fgm_home': 'fgm', 'fga_home': 'fga',
    'fg3m_home': 'fg3m', 'fg3a_home': 'fg3a', 'ftm_home': 'ftm', 'fta_home': 'fta',
    'reb_home': 'reb', 'ast_home': 'ast'})
home_df['is_home'] = 1

away_df = games[away_cols].copy().rename(columns={
    'team_id_away': 'team_id', 'pts_away': 'pts', 'fgm_away': 'fgm', 'fga_away': 'fga',
    'fg3m_away': 'fg3m', 'fg3a_away': 'fg3a', 'ftm_away': 'ftm', 'fta_away': 'fta',
    'reb_away': 'reb', 'ast_away': 'ast'})
away_df['is_home'] = 0
#construye un dataset por equipo uniendo los resultados como local y visitante
team_games = pd.concat([home_df, away_df], ignore_index=True).sort_values(['team_id', 'game_date']).reset_index(drop=True)

team_games['win'] = 0
for idx, row in team_games.iterrows():
    if row['is_home'] == 1:
        result = games.loc[games['game_id'] == row['game_id'], 'home_win'].values[0]
    else:
        result = 1 - games.loc[games['game_id'] == row['game_id'], 'home_win'].values[0]
    team_games.at[idx, 'win'] = result

ewma_stats = ['pts', 'fgm', 'fga', 'fg3m', 'fg3a', 'ftm', 'fta', 'reb', 'ast', 'win']
ewma_list = []
for team, grp in team_games.groupby('team_id', sort=False):
    #calcula las medias exponenciales de las stats. Cada partido ve la media ponderada de 
    #los ultimos resultados del equipo, dandole mas peso a los ultimos resultados
    grp = grp.sort_values('game_date').copy()
    grp['streak5'] = grp['win'].rolling(5, min_periods=1).mean().shift(1)
    ewm = grp[ewma_stats].ewm(span=EWMA_SPAN, adjust=False).mean().shift(1)
    ewm.columns = [f'ewm_{c}' for c in ewma_stats]
    out = pd.concat([grp.reset_index(drop=True), ewm.reset_index(drop=True)], axis=1)
    ewma_list.append(out)
team_games_ewm = pd.concat(ewma_list, ignore_index=True)

rename_home = {f'ewm_{s}': f'home_ewm_{s}' for s in ewma_stats}
rename_home['streak5'] = 'home_streak5'
rename_away = {f'ewm_{s}': f'away_ewm_{s}' for s in ewma_stats}
rename_away['streak5'] = 'away_streak5'
#guarda el estado de forma del equipo
home_ewm = team_games_ewm[team_games_ewm['is_home'] == 1][['game_id', 'streak5'] + [f'ewm_{s}' for s in ewma_stats]].rename(columns=rename_home)
away_ewm = team_games_ewm[team_games_ewm['is_home'] == 0][['game_id', 'streak5'] + [f'ewm_{s}' for s in ewma_stats]].rename(columns=rename_away)

games = games.merge(home_ewm, on='game_id', how='left')
games = games.merge(away_ewm, on='game_id', how='left')


In [6]:
#5:Features finales
#une todas las variables cradas antes en los puntos 2,3 y 4 de cara a alimentar el modelo
games['fg_eff_home'] = games['home_ewm_fgm'] / (games['home_ewm_fga'] + 1e-6)
games['fg_eff_away'] = games['away_ewm_fgm'] / (games['away_ewm_fga'] + 1e-6)
games['ft_eff_home'] = games['home_ewm_ftm'] / (games['home_ewm_fta'] + 1e-6)
games['ft_eff_away'] = games['away_ewm_ftm'] / (games['away_ewm_fta'] + 1e-6)

games['streak_diff'] = games['home_streak5'] - games['away_streak5']
games['fg_eff_diff'] = games['fg_eff_home'] - games['fg_eff_away']
games['ft_eff_diff'] = games['ft_eff_home'] - games['ft_eff_away']
games['reb_diff'] = games['home_ewm_reb'] - games['away_ewm_reb']
games['ast_diff'] = games['home_ewm_ast'] - games['away_ewm_ast']

feat_cols = [
    'elo_diff', 'streak_diff', 'fg_eff_diff', 'ft_eff_diff', 'reb_diff', 'ast_diff', 'diff_pts_line'
]
games[feat_cols] = games[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0)


NameError: name 'games' is not defined

In [7]:
#6:Split
#entrena con los partidos de antes de 2023
#testea los partidos de 2023
cut = pd.Timestamp('2023-01-01')
train = games[games['game_date'] < cut].copy()
test = games[(games['game_date'] >= cut) & (games['game_date'] < pd.Timestamp('2024-01-01'))].copy()

X_train = train[feat_cols].values
y_train = train['home_win'].values
X_test = test[feat_cols].values
y_test = test['home_win'].values
#se usa para dar mas peso a las ultimas temporadas, lo hace con el decay_factor
years_diff = (cut - train['game_date']).dt.days / 365
sample_weight = DECAY_FACTOR ** years_diff


NameError: name 'games' is not defined

In [ ]:
#7:Modelos
#usa los modelos XGBClassifier,RandomForestClassifier y LogisticRegression combinandolos
xgb = XGBClassifier(
    n_estimators=350, learning_rate=0.03, max_depth=6,
    subsample=0.9, colsample_bytree=0.9,
    random_state=RANDOM_STATE, eval_metric='logloss'
)
xgb_cal = CalibratedClassifierCV(xgb, cv=3, method='sigmoid')
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
#El sample_weight con decaimiento temporal se aplica a todos los modelos del ensemble, logrando que las temporadas viejas influyan menos
xgb_cal.fit(X_train, y_train, sample_weight=sample_weight)
voting = VotingClassifier(
    estimators=[('xgb', xgb_cal), ('rf', rf), ('lr', lr)],
    voting='soft', n_jobs=-1
)
voting.fit(X_train, y_train, sample_weight=sample_weight)


In [ ]:
#8:Predicción
probs = voting.predict_proba(X_test)[:, 1]#probablidad de que gane el local
preds = (probs > 0.5).astype(int)

#calcula las metricas
acc = accuracy_score(y_test, preds)
print(f"\n Accuracy 2023 mejorado: {acc*100:.2f}%")
print(classification_report(y_test, preds, target_names=['VISITANTE', 'LOCAL']))

#genera una metriz visual para ver los aciertos y errores entre locales y visitantes
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de Confusión - Predicción 2023")
plt.show()


In [ ]:
#9:Export:guarda los resultados en un .csv
test_out = test[['game_id', 'game_date', 'team_id_home', 'team_id_away', 'pts_home', 'pts_away', 'home_win']].copy()
test_out['prob_local'] = probs
test_out['winner_pred'] = np.where(preds == 1, 'LOCAL', 'VISITANTE')
test_out['acierto'] = (preds == test_out['home_win']).astype(int)
test_out.to_csv('predicciones_2023_mejorado.csv', index=False)
print("✅ CSV guardado: predicciones_2023_mejorado.csv")